# Workflows et Human-in-the-Loop — Jour 3

Après les bases des agents et de LangChain, nous allons construire un graphe avec
LangGraph pour maîtriser ses étapes et y intégrer une intervention humaine
(**Human-in-the-Loop**, ou **HITL**).

L'atelier part d'une validation simple, ajoute le routage et la persistance, puis
les réunit dans un **workflow de support email avec LLM et Human-in-the-Loop**.
Vous apprendrez à suspendre et reprendre une exécution, à réserver la revue humaine
aux cas sensibles et à distinguer audit métier et historique technique.

## 1. Comprendre un workflow
### 1.1 Agent ou workflow ?

Un **agent** décide dynamiquement des actions à effectuer, tandis qu'un **workflow**
définit explicitement les étapes et les transitions. Un workflow peut comporter
des branches et des boucles : ses règles déterminent les chemins possibles.

Un appel à un **LLM** (grand modèle de langage) ne suffit donc pas à faire un agent.
Dans notre workflow de support, le modèle classe le message et rédige une réponse ;
le code organise les étapes et décide quand demander une validation humaine.
Un workflow peut aussi intégrer un agent dans un de ses nœuds.

![Comparaison entre workflows et agent](../assets/agent_workflow.png)

Le schéma illustre des chemins organisés par le code, à gauche, et une boucle où
le modèle choisit les actions, à droite. Source :
[documentation LangGraph](https://docs.langchain.com/oss/python/langgraph/workflows-agents).

### 1.2 Les repères essentiels

- **État (`state`)** : données partagées entre les étapes du graphe.
- **Nœud (`node`)** : fonction qui lit l'état et renvoie des mises à jour.
- **Arête (`edge`)** : transition entre deux nœuds. `StateGraph` assemble ces éléments.
- **Interruption et reprise** : `interrupt()` suspend le graphe ; `Command(resume=...)` lui fournit une réponse pour continuer.
- **Checkpointer** : composant qui sauvegarde des instantanés de l'état, appelés checkpoints. Le `thread_id` identifie le fil d'exécution à retrouver lors de la reprise.

Ces notions prennent forme dans le premier exemple : une étape de validation entre
le début (`START`) et la fin (`END`) du graphe.

### 1.3 Dépendances et portabilité

Utilisez le noyau Python de l'environnement de formation, qui fournit notamment
`langgraph`, `langchain-mistralai` et `python-dotenv`. Ajoutez le composant SQLite si nécessaire :

```powershell
uv pip install langgraph-checkpoint-sqlite
```

Certaines cellules supposent que le répertoire de travail est celui du notebook :
`env_utils` vient de `../util`, la configuration de `../.env` et l'image de
`../assets/agent_workflow.png`. Hors de l'environnement fourni, adaptez ces chemins
et renseignez `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL`.

### 1.4 Configurer le modèle

Nous chargeons les variables d'environnement et ajustons l'URL du serveur pour
qu'elle se termine par `/v1`. Le modèle sera utilisé dans le projet de support ;
les premiers graphes illustrent les mécanismes sans appel au LLM.

**Objectif.** Préparer les paramètres de connexion au serveur Mistral.

**Méthode.** Charger le fichier `.env`, construire l'URL terminée par `/v1`, puis utiliser l'utilitaire de formation pour contrôler les variables chargées.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

# Charger les variables du fichier .env sans remplacer celles déjà définies.
load_dotenv(find_dotenv())

# Éviter de doubler le suffixe /v1 lorsque le serveur le fournit déjà.
_base = os.environ["MISTRAL_SERVER_URL"].rstrip("/")
MISTRAL_ENDPOINT = _base if _base.endswith("/v1") else _base + "/v1"

import sys
# Rendre accessible l’utilitaire fourni dans le répertoire de formation.
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "util"))
from env_utils import doublecheck_env

doublecheck_env("../.env")

MISTRAL_API_KEY=****uLt0
MISTRAL_SERVER_URL=****l.ai
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=false
LANGSMITH_PROJECT=****tral


**Après exécution — résultat attendu.** `MISTRAL_ENDPOINT` contient l'URL à utiliser par le client. L'utilitaire affiche un contrôle des variables, avec les valeurs masquées, ou signale l'absence du fichier `../.env`. Aucun appel au modèle n'a encore lieu. Une erreur sur `MISTRAL_SERVER_URL` indique que cette variable doit être renseignée avant de continuer.

**Objectif.** Créer le client LLM qui servira à classer les emails et à rédiger les réponses.

**Méthode.** Instancier `ChatMistralAI` avec le modèle choisi, l'URL préparée et une température de zéro pour limiter la variabilité des réponses.

In [2]:
from langchain_mistralai import ChatMistralAI

MODEL = "mistral-medium-latest"

# Créer le client ; les requêtes seront envoyées lors des appels à invoke().
llm = ChatMistralAI(
    model=MODEL,
    temperature=0,
    endpoint=MISTRAL_ENDPOINT,
)

print(f"LLM : {MODEL} · temperature=0")

LLM : mistral-medium-latest · temperature=0


**Après exécution — résultat attendu.** La ligne `LLM : mistral-medium-latest · temperature=0` confirme la création du client `llm`. Elle ne confirme pas encore l'accès au serveur : le premier appel au modèle aura lieu dans le scénario de support.

## 2. Construire un workflow avec interruption

Le premier graphe contient un seul nœud : il demande l'autorisation d'une action
et produit un résultat. L'action est simulée : aucun dossier n'est supprimé.

Voici le cycle à observer :

1. Le graphe exécute le nœud de validation.
2. `interrupt()` suspend l'exécution et expose la demande à l'application appelante.
3. Le checkpointer sauvegarde l'état nécessaire à la reprise.
4. L'application attend une intervention humaine ; ici, nous fournissons la réponse dans le code.
5. `Command(resume=...)` reprend le workflow sur le même `thread_id`.

Lors de la reprise, le nœud repart de son début et `interrupt()` renvoie la réponse
fournie. Évitez donc tout appel externe non idempotent avant l'interruption : une
opération qui crée un nouvel effet à chaque appel pourrait se répéter. Placez ces
effets après l'interruption ou dans un nœud distinct.

**Objectif.** Construire le plus petit workflow capable de demander une autorisation.

**Méthode.** Décrire l'état avec `MiniState`, définir le nœud de validation, puis relier `START`, `valider` et `END`. Compiler le graphe avec un checkpointer en mémoire pour permettre sa reprise.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


# Le nœud reçoit une action et complète ensuite le champ resultat.
class MiniState(TypedDict):
    action: str
    resultat: str


def demander_validation(state: MiniState) -> MiniState:
    # interrupt() suspend le graphe et renvoie ce dictionnaire à l'appelant.
    reponse = interrupt({"action": state["action"], "question": "Autoriser ? (oui/non)"})
    # À la reprise, reponse prend la valeur transmise dans Command(resume=...).
    if reponse == "oui":
        return {"resultat": f" '{state['action']}' exécutée"}
    return {"resultat": f" '{state['action']}' annulée"}


# Définir le parcours unique : début, validation, fin.
mini = StateGraph(MiniState)
mini.add_node("valider", demander_validation)
mini.add_edge(START, "valider")
mini.add_edge("valider", END)

# Conserver le checkpoint en mémoire pour retrouver la validation en attente.
mini_app = mini.compile(checkpointer=InMemorySaver())
print("Graphe minimal compilé")

Graphe minimal compilé


**Après exécution — résultat attendu.** Le message `Graphe minimal compilé` indique que `mini_app` est prêt. La compilation prépare le workflow ; elle n'exécute pas le nœud et ne demande donc encore aucune validation.

**Objectif.** Observer une interruption, puis reprendre le workflow avec un refus.

**Méthode.** Lancer `mini_app` sur le fil `mini-1`, lire la demande exposée dans `__interrupt__`, puis fournir `"non"` avec `Command(resume=...)` en conservant la même configuration.

In [4]:
# Réutiliser cet identifiant permet de retrouver la même attente à la reprise.
config = {"configurable": {"thread_id": "mini-1"}}

# 1) Lancement : le graphe s'arrête sur l'interruption.
resultat = mini_app.invoke({"action": "supprimer_dossier"}, config)

# Le contenu transmis à interrupt() est accessible dans le résultat de invoke().
interruption = resultat["__interrupt__"][0]
print("⏸  Interruption reçue :", interruption.value)

# 2) Décision humaine → reprise sur le MÊME thread_id.
resultat = mini_app.invoke(Command(resume="non"), config)
print("▶  Après reprise :", resultat["resultat"])

⏸  Interruption reçue : {'action': 'supprimer_dossier', 'question': 'Autoriser ? (oui/non)'}
▶  Après reprise :  'supprimer_dossier' annulée


**Après exécution — résultat attendu.** Deux affichages se succèdent : la demande d'autorisation, puis le résultat indiquant que `'supprimer_dossier'` est annulée. La seconde invocation a repris l'attente de la première ; aucune suppression réelle n'est effectuée.

La première invocation expose la demande dans `__interrupt__`. La seconde fournit
la réponse `"non"` : le résultat doit indiquer que l'action est annulée.
La reprise repose sur trois éléments : le checkpointer, le même `thread_id` et
la réponse transmise par `Command(resume=...)`.

## 3. Router après la décision humaine

Reprendre ne signifie pas toujours suivre le même chemin. Pour une revue de document,
approuver ou éditer conduit à la finalisation ; rejeter conduit à l'archivage.

`Command` permet à un nœud de combiner une **mise à jour de l'état** (`update`)
et une **décision de routage** (`goto`). Nous avons déjà utilisé son autre rôle :
reprendre un workflow interrompu avec `resume`, passé à `invoke()`.

Nous ajoutons aussi un journal `audit`. Son **reducer**, c'est-à-dire sa règle de
fusion, est `operator.add` : les nouvelles entrées s'ajoutent à la liste existante
au lieu de la remplacer.

**Objectif.** Transformer la décision humaine en choix de parcours tout en conservant une trace métier.

**Méthode.** Définir trois décisions dans `revue_humaine`. Chaque `Command` ajoute une entrée d'audit et choisit `finaliser` ou `archiver` ; la décision d'édition met aussi à jour le document. Le reducer concatène les entrées du journal.

In [5]:
import operator
from typing import Annotated
from datetime import datetime, timezone


class RevueState(TypedDict):
    document: str
    verdict: str
    audit: Annotated[list[dict], operator.add]  # reducer : concatène les entrées


def _entree_audit(decision: str, doc: str) -> dict:
    # Horodater la décision et conserver un extrait, sans recopier tout le document.
    return {
        "decision": decision,
        "horodatage": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "extrait": doc[:40],
    }


def revue_humaine(state: RevueState) -> Command:
    decision = interrupt({
        "document": state["document"],
        "action": "Relire : approuver / editer / rejeter",
    })
    # decision est un dict fourni par l'humain, ex. {"choix": "editer", "texte": "..."}
    choix = decision.get("choix")

    if choix == "approuver":
        return Command(update={"audit": [_entree_audit("approuvé", state["document"])]},
                       goto="finaliser")
    # Enregistrer le texte corrigé et l’audit avant de passer à la finalisation.
    if choix == "editer":
        nouveau = decision.get("texte", state["document"])
        return Command(update={"document": nouveau,
                               "audit": [_entree_audit("édité", nouveau)]},
                       goto="finaliser")
    # Tout choix autre qu’approuver ou éditer conduit ici au rejet.
    return Command(update={"audit": [_entree_audit("rejeté", state["document"])]},
                   goto="archiver")


def finaliser(state: RevueState) -> RevueState:
    return {"verdict": "publié"}


def archiver(state: RevueState) -> RevueState:
    return {"verdict": "archivé (rejeté)"}


revue = StateGraph(RevueState)
revue.add_node("revue_humaine", revue_humaine)
revue.add_node("finaliser", finaliser)
revue.add_node("archiver", archiver)
revue.add_edge(START, "revue_humaine")
# Le nœud de revue choisit sa destination avec Command : aucune arête fixe en sortie.
revue.add_edge("finaliser", END)
revue.add_edge("archiver", END)

revue_app = revue.compile(checkpointer=InMemorySaver())
print("Graphe de revue compilé (3 décisions, 2 destinations : finaliser / archiver)")

Graphe de revue compilé (3 décisions, 2 destinations : finaliser / archiver)


**Après exécution — résultat attendu.** Le message de compilation annonce trois décisions et deux destinations. `revue_app` est prêt à recevoir un document ; le verdict et l'audit ne seront produits qu'après une décision humaine.

**Objectif.** Vérifier qu'une correction humaine est appliquée avant la finalisation du document.

**Méthode.** Soumettre une note, récupérer la demande de relecture, puis reprendre le même fil avec le choix `editer` et un nouveau texte. Afficher le verdict, le document et le journal pour suivre l'effet de cette décision.

In [6]:
# Scénario : corriger le document, puis le finaliser dans le même parcours.
cfg = {"configurable": {"thread_id": "revue-edit"}}

etat = revue_app.invoke({"document": "Note de service : congés fermés en août."}, cfg)
print("⏸ ", etat["__interrupt__"][0].value["action"])

decision = {"choix": "editer", "texte": "Note : les congés d'août sont soumis à validation."}
# Injecter la décision dans l’interruption du même fil d’exécution.
final = revue_app.invoke(Command(resume=decision), cfg)

print("Verdict :", final["verdict"])
print("Document :", final["document"])
print("Audit   :", final["audit"])

⏸  Relire : approuver / editer / rejeter
Verdict : publié
Document : Note : les congés d'août sont soumis à validation.
Audit   : [{'decision': 'édité', 'horodatage': '2026-09-08T14:49:04+00:00', 'extrait': "Note : les congés d'août sont soumis à v"}]


**Après exécution — résultat attendu.** Le verdict doit être `publié`, le document doit devenir `Note : les congés d'août sont soumis à validation.` et l'audit doit contenir une entrée `édité`, avec un horodatage et un extrait du texte corrigé.

Observez que la correction du document et l'entrée d'audit sont produites après
`interrupt()`. Elles utilisent ainsi la décision reçue à la reprise. Le code placé
avant l'interruption peut être réexécuté : il doit pouvoir l'être sans effet indésirable.

## 4. Conserver l'état avec SQLite

Nous savons reprendre une validation, mais `InMemorySaver` conserve les checkpoints
uniquement en mémoire. Un arrêt du noyau Python les fait disparaître. `SqliteSaver`
les écrit dans un fichier local pour les retrouver après un redémarrage.

Les deux cellules suivantes ferment puis rouvrent la connexion au même fichier et
recompilent le graphe avec un nouveau checkpointer. Elles illustrent une reprise
depuis le disque, sans réellement redémarrer le processus Python. Après un véritable
redémarrage, il faut aussi recharger les définitions du graphe et utiliser le même
fichier et le même `thread_id`.

Pour repartir de zéro en atelier, la première cellule supprime la base de démonstration
si elle existe. Ne relancez pas cette initialisation pour reprendre une attente à conserver.

**Objectif.** Sauvegarder une validation en attente dans une base locale.

**Méthode.** Initialiser la base de démonstration, compiler le graphe de revue avec `SqliteSaver`, puis soumettre un document. L'appel s'interrompt avant la décision humaine ; fermer ensuite la connexion SQLite.

In [7]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

DB = "checkpoints_t1.db"
if os.path.exists(DB):
    os.remove(DB)  # Réinitialisation de démonstration : les anciennes attentes sont effacées.
cfg_durable = {"configurable": {"thread_id": "revue-durable"}}

# Première connexion : sauvegarder le graphe interrompu sur disque.
# Autoriser l’usage de la connexion par les threads d’exécution du graphe.
conn1 = sqlite3.connect(DB, check_same_thread=False)
app1 = revue.compile(checkpointer=SqliteSaver(conn1))
# L’exécution atteint interrupt() et sauvegarde l’attente avant de rendre la main.
app1.invoke({"document": "Contrat fournisseur à signer."}, cfg_durable)
print("Première connexion : graphe interrompu, état écrit sur disque.")
conn1.close()  # ferme la connexion ; le processus Python reste actif

Première connexion : graphe interrompu, état écrit sur disque.


**Après exécution — résultat attendu.** Le message indique que le graphe est interrompu et l'état écrit sur disque. Le fichier `checkpoints_t1.db` conserve cette attente après la fermeture de la connexion. La cellule suivante vérifiera qu'une nouvelle connexion peut la retrouver.

**Objectif.** Retrouver l'attente sauvegardée et terminer la revue depuis une nouvelle connexion.

**Méthode.** Rouvrir le même fichier, recompiler le même graphe, puis consulter son état avec `get_state()` et le même `thread_id`. Reprendre avec une approbation avant de fermer cette seconde connexion.

In [8]:
# Nouvelle connexion : retrouver la même attente depuis le fichier.
conn2 = sqlite3.connect(DB, check_same_thread=False)
app2 = revue.compile(checkpointer=SqliteSaver(conn2))

# L'état est reconstruit depuis le disque, pas depuis la mémoire.
etat_repris = app2.get_state(cfg_durable)
print("Nœud en attente :", etat_repris.next)  # ('revue_humaine',)

# Le même thread_id associe cette approbation à la validation sauvegardée.
final = app2.invoke(Command(resume={"choix": "approuver"}), cfg_durable)
print("Verdict après reprise :", final["verdict"])
print("Audit :", final["audit"])
conn2.close()

Nœud en attente : ('revue_humaine',)
Verdict après reprise : publié
Audit : [{'decision': 'approuvé', 'horodatage': '2026-09-08T14:49:04+00:00', 'extrait': 'Contrat fournisseur à signer.'}]


**Après exécution — résultat attendu.** Le nœud en attente doit être `('revue_humaine',)`. Après reprise, le verdict doit être `publié` et l'audit doit contenir la décision `approuvé`. Ces sorties montrent que l'attente a été retrouvée depuis le fichier ; elles ne simulent pas un redémarrage réel du noyau.

Comme repères de progression, `InMemorySaver` convient aux démonstrations,
SQLite à une persistance locale sur une machine, et un backend externe adapté,
par exemple PostgreSQL, peut répondre aux besoins d'un déploiement distribué.
Ce ne sont pas des règles absolues : le choix dépend notamment de la concurrence,
du volume et de l'exploitation. La logique du graphe reste indépendante du stockage.

## 5. Construire le workflow de support email

Réunissons les mécanismes précédents : le workflow lit un email, le classe, collecte
du contexte, rédige une réponse puis l'envoie, avec une revue humaine si nécessaire.
La classification et la rédaction utilisent Mistral ; la recherche documentaire,
la création du ticket et l'envoi sont simulés.

### 5.1 Définir l'état et les étapes

```text
START → read_email → classify_intent
                         ├→ search_documentation ─┐
                         └→ bug_tracking ─────────┴→ write_response
                                                        ├→ send_reply → END
                                                        └→ human_review
                                                              ├→ send_reply → END
                                                              └→ END (rejet)
```

La classification utilise `with_structured_output` pour obtenir les champs du
schéma Pydantic ci-dessous. Ces valeurs alimentent ensuite la règle de validation.

**Objectif.** Définir les données dont le workflow de support aura besoin à chaque étape.

**Méthode.** Utiliser un modèle Pydantic pour encadrer la classification du LLM et un `TypedDict` pour décrire l'état partagé. Séparer les données d'entrée, le contexte collecté, la réponse et l'audit.

In [9]:
from typing import Literal, Optional
from pydantic import BaseModel, Field


class EmailClassification(BaseModel):
    """Schéma que le modèle Mistral doit remplir pour classer l'email."""
    # Limiter les catégories permet au code de routage de tester des valeurs connues.
    intent: Literal["question", "bug", "facturation", "fonctionnalite", "complexe"] = Field(
        description="Intention principale de l'email"
    )
    urgency: Literal["basse", "moyenne", "haute", "critique"] = Field(
        description="Niveau d'urgence"
    )
    topic: str = Field(description="Sujet en quelques mots")
    summary: str = Field(description="Résumé en une phrase")


class EmailState(TypedDict):
    # Entrée
    email_content: str
    sender_email: str
    email_id: str
    # Champs renseignés progressivement par les nœuds de classification et de recherche.
    classification: Optional[dict]
    ticket_id: Optional[str]
    search_results: Optional[list]
    # Sortie
    draft_response: Optional[str]
    sent: Optional[bool]
    # Ajouter les nouvelles décisions sans écraser le journal existant.
    audit: Annotated[list[dict], operator.add]


print("Schémas définis")

Schémas définis


**Après exécution — résultat attendu.** Le message `Schémas définis` confirme que les deux classes sont disponibles. Aucun email n'est encore classé. `EmailClassification` encadre la sortie du modèle ; `EmailState` décrit les champs que les nœuds alimenteront progressivement.

### 5.2 Choisir quand demander une validation

Le HITL est ici une **règle métier du workflow**, appliquée après la rédaction :

- urgence basse ou moyenne : traitement automatique, sauf demande complexe ;
- urgence haute ou critique : validation humaine ;
- demande complexe : validation humaine, quelle que soit l'urgence.

L'humain intervient lorsque le risque ou la complexité le justifie, sans devoir
valider chaque étape. Dans `write_response`, `besoin_revue` applique cette règle
à la classification du modèle ; seul `human_review` déclenche l'interruption.

**Objectif.** Définir les étapes du support et traduire la politique HITL en code.

**Méthode.** Préparer un appel de classification structuré, deux fonctions de collecte du contexte et une fonction de rédaction. À partir de la classification, router vers la revue humaine ou l'envoi simulé. Après une revue, conserver la décision dans l'audit et appliquer la correction éventuelle.

In [10]:
import uuid

# Préparer des réponses conformes au schéma ; aucun appel réseau n’est encore lancé.
structured_llm = llm.with_structured_output(EmailClassification)


def read_email(state: EmailState) -> EmailState:
    # Ici, un vrai connecteur (IMAP/API) analyserait l'email. On passe l'entrée telle quelle.
    return {}


def classify_intent(state: EmailState) -> EmailState:
    prompt = (
        "Classe cet email de support client.\n\n"
        f"Expéditeur : {state['sender_email']}\n"
        f"Email : {state['email_content']}"
    )
    classification = structured_llm.invoke(prompt)
    # On stocke un dict (sérialisable par le checkpointer).
    return {"classification": classification.model_dump()}


def search_documentation(state: EmailState) -> EmailState:
    # Recherche documentaire simulée ; le thème suivant ajoutera des sources réelles.
    sujet = (state.get("classification") or {}).get("topic", "")
    return {"search_results": [f"[doc] Article pertinent sur : {sujet}"]}


def bug_tracking(state: EmailState) -> EmailState:
    # Simuler un identifiant de ticket uniquement pour les bugs ; aucun outil externe n’est appelé.
    if (state.get("classification") or {}).get("intent") == "bug":
        return {"ticket_id": f"BUG-{uuid.uuid4().hex[:8]}"}
    return {}


def write_response(state: EmailState) -> Command[Literal["human_review", "send_reply"]]:
    c = state.get("classification") or {}
    # Réunir les résultats des deux branches pour préparer la réponse.
    contexte = "\n".join(state.get("search_results") or [])
    ticket = f"\nTicket ouvert : {state['ticket_id']}" if state.get("ticket_id") else ""

    prompt = (
        "Rédige une réponse de support professionnelle, en français, à cet email.\n\n"
        f"Email : {state['email_content']}\n"
        f"Résumé : {c.get('summary')}\n"
        f"Contexte documentaire :\n{contexte}{ticket}"
    )
    reponse = llm.invoke(prompt)

    # Règle métier : une urgence élevée ou une demande complexe impose une revue.
    besoin_revue = c.get("urgency") in ("haute", "critique") or c.get("intent") == "complexe"
    return Command(
        update={"draft_response": reponse.content},
        goto="human_review" if besoin_revue else "send_reply",
    )


def human_review(state: EmailState) -> Command[Literal["send_reply", "__end__"]]:
    c = state.get("classification") or {}
    # Suspendre ici pour exposer le brouillon ; decision sera fourni lors de la reprise.
    decision = interrupt({
        "email_id": state["email_id"],
        "urgency": c.get("urgency"),
        "intent": c.get("intent"),
        "draft_response": state.get("draft_response", ""),
        "action": "Relire puis approuver (avec édition possible) ou rejeter",
    })

    # Écrire la trace après la reprise, lorsque la décision humaine est disponible.
    entree = {
        "email_id": state["email_id"],
        "decision": "approuvé" if decision.get("approved") else "rejeté",
        "horodatage": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if decision.get("approved"):
        return Command(
            update={
                # Garder le brouillon initial si aucune correction non vide n’est fournie.
                "draft_response": decision.get("edited_response") or state.get("draft_response", ""),
                "audit": [entree],
            },
            goto="send_reply",
        )
    # Un rejet termine le graphe sans passer par le nœud d’envoi.
    return Command(update={"audit": [entree], "sent": False}, goto=END)


def send_reply(state: EmailState) -> EmailState:
    # L'envoi est simulé ; un connecteur réel serait appelé dans ce nœud.
    print(f" Envoi simulé à {state['sender_email']} (email {state['email_id']})")
    return {"sent": True}


print("Nœuds définis")

Nœuds définis


**Après exécution — résultat attendu.** Le message `Nœuds définis` confirme que les fonctions sont disponibles. Elles ne sont pas encore exécutées : aucun email n'a été traité. Repérez `besoin_revue`, qui choisit le parcours, et `interrupt()`, qui suspend réellement la revue humaine lorsque ce nœud est atteint.

### 5.3 Paralléliser puis synchroniser

Après `classify_intent`, le **fan-out** lance les deux branches en parallèle :
la recherche documentaire et le suivi des bugs, qui ne produit un ticket que si
l'intention vaut `bug`. Les branches écrivent dans des champs distincts de l'état.

Le **fan-in** les synchronise : l'arête qui prend la liste des deux nœuds impose
que `write_response` attende leur fin. La rédaction dispose ainsi de tout le contexte.
Les décisions suivantes sont prises par `Command(goto=...)`.

La base `emails_t1.db` est réinitialisée pour la démonstration. Relancer cette cellule
efface donc les attentes et l'historique des scénarios précédents.

**Objectif.** Assembler les nœuds en un workflow qui attend les deux branches avant de rédiger.

**Méthode.** Enregistrer les nœuds, ajouter les transitions et la jonction explicite des deux branches, puis compiler avec un checkpointer SQLite. Afficher la description Mermaid pour examiner les connexions.

In [11]:
# Enregistrer les fonctions sous les noms utilisés dans les transitions.
builder = StateGraph(EmailState)
builder.add_node("read_email", read_email)
builder.add_node("classify_intent", classify_intent)
builder.add_node("search_documentation", search_documentation)
builder.add_node("bug_tracking", bug_tracking)
builder.add_node("write_response", write_response)
builder.add_node("human_review", human_review)
builder.add_node("send_reply", send_reply)

builder.add_edge(START, "read_email")
builder.add_edge("read_email", "classify_intent")
# Fan-out : les deux recherches sont lancées en parallèle.
builder.add_edge("classify_intent", "search_documentation")
builder.add_edge("classify_intent", "bug_tracking")
# Fan-in : la rédaction attend la fin des deux branches.
builder.add_edge(
    ["search_documentation", "bug_tracking"],
    "write_response"
)
builder.add_edge("send_reply", END)
# write_response et human_review routent via Command(goto=...), pas d'arête statique.

if os.path.exists("emails_t1.db"):
    os.remove("emails_t1.db")  # départ propre si le notebook est relancé
conn_email = sqlite3.connect("emails_t1.db", check_same_thread=False)
# Sauvegarder les attentes humaines dans la base locale.
email_app = builder.compile(checkpointer=SqliteSaver(conn_email))

# Afficher la description du graphe sans lancer de traitement d’email.
print(email_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	read_email(read_email)
	classify_intent(classify_intent)
	search_documentation(search_documentation)
	bug_tracking(bug_tracking)
	write_response(write_response)
	human_review(human_review)
	send_reply(send_reply)
	__end__([<p>__end__</p>]):::last
	__start__ --> read_email;
	bug_tracking --> write_response;
	classify_intent --> bug_tracking;
	classify_intent --> search_documentation;
	human_review -.-> __end__;
	human_review -.-> send_reply;
	read_email --> classify_intent;
	search_documentation --> write_response;
	write_response -.-> human_review;
	write_response -.-> send_reply;
	send_reply --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



**Après exécution — résultat attendu.** La sortie est du texte au format Mermaid, pas une image. On doit y retrouver les deux branches issues de `classify_intent`, leur arrivée sur `write_response` et les destinations de routage. La liste passée à `add_edge` garantit l'attente des deux branches ; le dessin seul ne prouve pas cette synchronisation. `email_app` est prêt, mais aucun email n'a encore été traité.

### 5.4 Observer un envoi automatique

Une question simple devrait être classée avec une urgence basse ou moyenne, puis
traitée sans validation humaine. Comparez la classification obtenue, la présence
d'une interruption et l'indicateur d'envoi simulé. Le routage dépend des valeurs
renvoyées par le modèle, pas du nom donné au scénario.

**Objectif.** Tester le parcours automatique sur une demande courante.

**Méthode.** Fournir un email de réinitialisation de mot de passe, un journal initialement vide et un identifiant de fil propre au dossier. Exécuter le workflow et lire la classification, l'indicateur d'interruption et l'état de l'envoi.

In [12]:
email_simple = {
    "email_content": "Bonjour, comment réinitialiser mon mot de passe ? Merci.",
    "sender_email": "client@exemple.fr",
    "email_id": "MAIL-1001",
    "audit": [],
}
# Un fil distinct évite de mélanger ce dossier avec les autres scénarios.
cfg1 = {"configurable": {"thread_id": "mail-1001"}}

# Cette invocation exécute les nœuds et effectue les appels au modèle.
res1 = email_app.invoke(email_simple, cfg1)
print("\nClassification :", res1["classification"])
print("Interruption ? :", "__interrupt__" in res1)
print("Envoyé ? :", res1.get("sent"))

 Envoi simulé à client@exemple.fr (email MAIL-1001)

Classification : {'intent': 'question', 'urgency': 'basse', 'topic': 'Réinitialisation du mot de passe', 'summary': 'Le client demande comment réinitialiser son mot de passe.'}
Interruption ? : False
Envoyé ? : True


**Après exécution — résultat attendu.** Si le modèle classe la demande avec une urgence basse ou moyenne et une intention autre que `complexe`, la sortie doit indiquer `Interruption ? : False` et `Envoyé ? : True`, avec un message d'envoi simulé. Si une interruption apparaît, examinez la classification : elle explique pourquoi la règle a demandé une revue.

### 5.5 Valider un cas sensible

Cet incident devrait recevoir une urgence haute ou critique. Le workflow prépare
alors un brouillon puis attend une décision humaine. Vérifiez la classification :
si le modèle ne déclenche pas la règle HITL, aucune reprise n'est nécessaire.

**Objectif.** Observer l'arrêt du workflow avant l'envoi d'une réponse à un incident sensible.

**Méthode.** Soumettre l'incident sur un fil distinct, puis vérifier la présence de `__interrupt__`. S'il existe, afficher l'urgence et les 500 premiers caractères du brouillon proposé à la relecture.

In [13]:
email_critique = {
    "email_content": (
        "URGENT : votre service est indisponible depuis 2 heures, "
        "nous perdons des ventes. C'est inacceptable, réagissez immédiatement !"
    ),
    "sender_email": "grand.compte@exemple.fr",
    "email_id": "MAIL-2002",
    "audit": [],
}
# Conserver cette configuration pour reprendre ensuite ce dossier précis.
cfg2 = {"configurable": {"thread_id": "mail-2002"}}

res2 = email_app.invoke(email_critique, cfg2)

# La décision de revue dépend de la classification réelle du modèle.
if "__interrupt__" in res2:
    demande = res2["__interrupt__"][0].value
    print("⏸  Validation demandée (urgence =", demande["urgency"], ")\n")
    print("Brouillon proposé :\n", demande["draft_response"][:500])
else:
    print("Envoyé sans revue :", res2.get("sent"))

⏸  Validation demandée (urgence = critique )

Brouillon proposé :
 **Objet : Réponse urgente – Indisponibilité du service (Ticket BUG-41118b35)**

Bonjour [Nom du client],

Nous vous remercions pour votre signalement et comprenons parfaitement l’urgence de la situation. Nous sommes sincèrement désolés pour cette indisponibilité et son impact sur votre activité.

Notre équipe technique a été immédiatement mobilisée pour investiguer le problème (réf. **Ticket BUG-41118b35**). Voici les actions en cours :
- **Diagnostic prioritaire** : Identification de la cause r


**Après exécution — résultat attendu.** Si la classification déclenche le HITL, une demande de validation et un extrait du brouillon apparaissent. L'envoi attend alors la décision de la cellule suivante. Sinon, la sortie indique un envoi sans revue : la classification du modèle n'a pas satisfait la règle d'interruption.

**Objectif.** Approuver le brouillon après l'avoir complété et terminer le dossier.

**Méthode.** Lorsqu'une interruption existe, ajouter un engagement de suivi au brouillon et transmettre l'approbation avec `Command(resume=decision)` sur `cfg2`. Sinon, conserver directement le résultat déjà terminé pour la suite de l'atelier.

In [14]:
if "__interrupt__" in res2:
    # Décision humaine : on approuve en éditant légèrement le brouillon.
    brouillon = res2["__interrupt__"][0].value["draft_response"]
    decision = {
        "approved": True,
        "edited_response": brouillon + "\n\nNous vous tenons informé toutes les 30 minutes.",
    }
    
    # Reprendre l’attente existante ; ne pas soumettre à nouveau l’email initial.
    final2 = email_app.invoke(Command(resume=decision), cfg2)
    print("Envoyé ? :", final2.get("sent"))
    print("Audit :", final2["audit"])
else:
    # Fournir un résultat final à la cellule d’audit même sans revue humaine.
    final2 = res2
    print("Aucune interruption : le workflow a déjà terminé.")

 Envoi simulé à grand.compte@exemple.fr (email MAIL-2002)
Envoyé ? : True
Audit : [{'email_id': 'MAIL-2002', 'decision': 'approuvé', 'horodatage': '2026-09-08T14:49:18+00:00'}]


**Après exécution — résultat attendu.** Après une reprise avec approbation, `Envoyé ? : True` et une entrée d'audit `approuvé` sont attendus. Le texte corrigé est conservé dans `final2['draft_response']` ; il n'est pas affiché par cette cellule. Sans interruption préalable, le message indique que le workflow a déjà terminé et aucune décision humaine n'est ajoutée.

Dans cet atelier, la décision humaine est un dictionnaire fourni dans le code.
Dans une application, une interface recueillerait l'approbation, le rejet ou la
correction du brouillon, puis transmettrait cette valeur à `Command(resume=...)`.

## 6. Distinguer audit métier et historique technique

L'**audit métier** conserve les décisions utiles au suivi du dossier : classification,
urgence, recours à une validation, décision finale et corrections humaines éventuelles.
Dans cet exemple volontairement simple, `audit` enregistre seulement l'approbation
ou le rejet et son horodatage. La classification et le brouillon corrigé restent
dans l'état ; ils ne constituent pas à eux seuls un journal métier complet.

L'**historique technique** rassemble les checkpoints et états successifs du graphe.
`get_state_history()` permet de les consulter pour comprendre les transitions et
déboguer. La cellule suivante les lit, du plus récent au plus ancien : elle ne
rejoue pas l'exécution. Les checkpoints peuvent aussi servir de point de départ
à une reprise ou à un rejeu explicite.

**Objectif.** Lire séparément les décisions métier et le déroulement technique du dossier sensible.

**Méthode.** Parcourir d'abord `final2['audit']`, puis les instantanés renvoyés par `get_state_history(cfg2)`. Pour chaque checkpoint, afficher la prochaine étape prévue, du plus récent au plus ancien.

In [15]:
# 1) Le journal d'audit métier (nos entrées explicites).
print("=== Journal d'audit (MAIL-2002) ===")
for entree in final2["audit"]:
    print(f"- {entree['horodatage']} · {entree['decision']} · {entree['email_id']}")

# 2) Consulter les checkpoints sans réexécuter le graphe.
print("\n=== Checkpoints traversés (du plus récent au plus ancien) ===")
for snap in email_app.get_state_history(cfg2):
    # Un tuple vide signifie qu’aucun nœud ne reste à exécuter à ce checkpoint.
    prochaine = snap.next or ("aucune étape en attente",)
    print(f"- prochaine étape : {prochaine}")

=== Journal d'audit (MAIL-2002) ===
- 2026-09-08T14:49:18+00:00 · approuvé · MAIL-2002

=== Checkpoints traversés (du plus récent au plus ancien) ===
- prochaine étape : ('aucune étape en attente',)
- prochaine étape : ('send_reply',)
- prochaine étape : ('human_review',)
- prochaine étape : ('write_response',)
- prochaine étape : ('search_documentation', 'bug_tracking')
- prochaine étape : ('classify_intent',)
- prochaine étape : ('read_email',)
- prochaine étape : ('__start__',)


**Après exécution — résultat attendu.** Après approbation, le premier bloc affiche une décision horodatée pour `MAIL-2002`. Sans revue humaine, ce journal peut être vide. Le second bloc affiche les étapes prévues aux différents checkpoints, avec `aucune étape en attente` pour un graphe terminé. Cette lecture ne rejoue aucun nœud et ne renvoie aucun email.

Retenez la différence : le journal métier raconte les décisions que vous choisissez
de tracer ; l'historique technique décrit l'exécution du graphe. Un historique de
checkpoints ne remplace donc pas un audit métier conçu pour les besoins du service.

## 7. Conclusion

Un workflow rend le chemin d'exécution explicite. LangGraph en organise l'état,
les transitions et les branches parallèles, dont le fan-in synchronise les résultats.
`interrupt()` suspend l'exécution ; un checkpointer persistant conserve cette attente
au-delà du processus. `Command` permet de reprendre ou de mettre à jour l'état et router.

L'intervention humaine devient une règle métier : elle se déclenche lorsque le
risque ou la complexité le justifie. L'audit métier explique les décisions ;
l'historique technique aide à comprendre et à reprendre l'exécution.

### 7.1 Pour aller plus loin

Remplacez la recherche simulée par le module documentaire du thème suivant, ou
enrichissez l'audit pour tracer la classification, le choix de revue et les corrections.
Un rejeu depuis un checkpoint antérieur est un exercice distinct : il peut réexécuter
des nœuds et leurs appels externes.

### 7.2 Ressources

- [LangGraph — Workflows et agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- [LangGraph — Interruptions et reprise](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [LangGraph — Persistance et checkpoints](https://docs.langchain.com/oss/python/langgraph/persistence)
- [Mistral — Appels de fonctions](https://docs.mistral.ai/capabilities/function_calling/)